# Zweiter Versuch: Skript fürs Training eines RL-Agenten zur Auftragssteuerung
## Hier: Schritt 1c, finetuning, trainiere das agv so, dass es sinnvolle Routen fährt
## Besonderheit hier: Die Befehle des AGV werden vorgefiltert, so dass fast nur sinnige Fahrten unternommen werden.
## Dann erst: Trainiere den Lager-Agenten, dass er sinnig auflädt unter der Annahme, dass das agv sinnige Routen fährt.

Der state wird für das AGV neu definiert, alles soll OHE sein:
4:  Erste vier Einträge: Anzahl geladener Waren OHE (0-4, wenn alle 4 Null dann logischerweise 0)
13: Die nächsten 13 Einträge OHE Standort AGV
4: Dann 4 Einträge, wieviel Waren an der Teststation abgeholt werden müssen, OHE (0-4, wenn alle 4 Null dann logischerweise 0)
13: Dann 13 Einträge als OHE-Maske, welche Stationen sinnvoll sind anzufahren (nach heuristischer Methodik: Nur die Stationen für die etwas geladen ist bzw. die Teststation nur dann, wenn genügend Platz für alle(!) abzuholenden Waren zur Verfügung steht).

In [1]:
import csv
import random
import numpy as np

##### Settting parameters ####

In [2]:
import factory_environment as env
import agv
import logic
import game_mechanics_4_agv_training as gm
import RL_Agent

In [3]:
my_env = env.env('ttable.csv', 'station_number.csv', 'env_control_order.csv')

In [4]:
my_agv = agv.agv()

In [5]:
my_logic = logic.logic()

In [6]:
gm = gm.game_mechanics(my_logic, my_env, my_agv)


## ⚙️ Reinforcement Learning Agent Parameters Explained

### 1. **gamma (γ) – Discount Factor**
- **Purpose**: Controls how much the agent values future rewards versus immediate ones.
- **Range**: Between 0 and 1.
- **Example**:
  - `γ = 0.99` → agent cares a lot about long-term rewards.
  - `γ = 0.1` → agent focuses mostly on immediate results.
- **Effect**: Higher values encourage farsightedness; lower values make the agent more short-term reactive.

---

### 2. **epsilon (ε) – Exploration Rate**
- **Purpose**: Determines the probability of the agent choosing a **random action** (exploration) versus the **best-known action** (exploitation).
- **Range**: Between 0 and 1.
- **Behavior**:
  - `ε = 1.0` → pure exploration (tries everything).
  - `ε = 0.01` → mostly exploits learned policy.
- **Why It Matters**: Without exploration, the agent might miss better strategies.

---

### 3. **epsilon_min**
- **Purpose**: Minimum bound for `epsilon` during training.
- **Prevents**: The agent from becoming completely deterministic too early.
- **Example**: `epsilon_min = 0.01` → agent will always have a 1% chance of trying something new.

---

### 4. **epsilon_decay**
- **Purpose**: Determines how quickly the `epsilon` value decays over time.
- **Example**:
  - `epsilon *= 0.995` per episode → slow decay.
  - `epsilon *= 0.9` → faster decay.
- **Goal**: Start explorative and gradually become confident in its knowledge.

---

### 5. **learning_rate (α)**
- **Purpose**: Determines how much new knowledge overrides old Q-values during updates.
- **Range**: Small positive float, like `0.001`.
- **Trade-off**:
  - High value: Learns fast, might oscillate or forget.
  - Low value: Stable but slow learning.

---

### 6. **memory (Replay Buffer)**
- **Type**: `deque` (double-ended queue) storing past experiences.
- **Why**: Helps decorrelate experiences for better training stability.
- **Stored Values**: Each memory is a tuple of `(state, action, reward, next_state, done)`.

---

### 7. **batch_size**
- **Purpose**: Number of experiences randomly sampled from memory to train on each time.
- **Effect**:
  - Small batch → more frequent updates, less stable.
  - Large batch → more stable updates, slower cycles.

---

### 8. **state_size**
- **What**: Number of features that describe the current environment state.
- **Use**: Defines the input shape of your neural network.

### 9. **action_size**
- **What**: Number of possible actions the agent can take.
- **Use**: Defines the output shape of your neural network.

---

Let me know if you'd like a cheat sheet summarizing this, or if you’re curious about how these interact during training. We can also dive deeper into things like target networks or dueling architectures when you're ready to advance! 🧠✨

In [ ]:
START_POS = 1   # Startet an Lager B

### Sonstige Training-Variablen
MAX_TIME_ONE_GAME = 3600*4
GAMES_PER_EPISODE = 25
GAMES_REPLAY = 10
EPISODE_COUNT = 50#400#500

#### Action size
# Fahre zu einer der 13 Stationen
action_size_agv = 13


#### Variablen für RL-Agent
GAMMA = 0.05  #0.99
EPSILON = 0.65 #0.9995
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.9995
LEARNING_RATE = 0.001*10
BATCH_SIZE = 512


In [8]:
loading = ''
my_RL_agent_agv = RL_Agent.DQNAgent(len(gm.act_state), action_size_agv, GAMMA, EPSILON, EPSILON_MIN, EPSILON_DECAY, LEARNING_RATE, loading)
my_RL_agent_agv.model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 34)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         4,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 13)             │           429 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,245 (59.55 KB)

 Trainable params: 15,245 (59.55 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
######
reward_vec = []
total_time_vec = []
gm.reset()
my_env.reset()
my_agv.reset()
for episode in range(EPISODE_COUNT):
    for game in range(GAMES_PER_EPISODE):
        for replays in range(GAMES_REPLAY):
            total_reward, total_time = gm.run_game(my_RL_agent_agv)
            reward_vec.append(total_reward)
            total_time_vec.append(total_time)
            my_env.reset()    
            my_agv.reset()
            gm.reset()        
        my_env.reset()
        gm.reset()
        #my_agv.reset()
    my_RL_agent_agv.replay(BATCH_SIZE)
    if episode % 5 == 0:
        print(f'epsilon was: {my_RL_agent_agv.epsilon}')
        print(f'mean reward episode {episode}: {np.mean(reward_vec[:-GAMES_PER_EPISODE+1])}')
        print(f'mean total time: {np.mean(total_time_vec[:-GAMES_PER_EPISODE+1])}')
    my_RL_agent_agv.epsilon *=EPSILON_DECAY
    
        

random decision
random decision
epsilon was: 0.8495750000000001
mean reward episode 0: 3.6145186321695264
mean total time: 2634.4194209278185
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
epsilon was: 0.8453366699862694
mean reward episode 5: 3.6267973902821247
mean total time: 2634.408571777496
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
random decision
epsilon was: 0.8411194840049142
mean reward episode 10: 3.658666751340971
mean total time: 2627.3787365533794
random decision
random decision
random decision
random decision
random decision

In [10]:
print(f'mittlere Belohnung für die ersten 100 Spiele: {np.mean(reward_vec[0:100])}')
print(f'mittlere Belohnung für die letzten 100 Spiele: {np.mean(reward_vec[:-100])}')
print(f'Abschließendes epsilon: {my_RL_agent_agv.epsilon}')

mittlere Belohnung für die ersten 100 Spiele: 3.641167027863777
mittlere Belohnung für die letzten 100 Spiele: 3.703693412036575
Abschließendes epsilon: 0.8085349007059838


In [11]:
mean_reward_per_episode = []
for i in range(0,len(reward_vec),GAMES_PER_EPISODE):
    if i % GAMES_PER_EPISODE == 0:
        mean_reward_per_episode.append(np.mean(reward_vec[i:i+GAMES_PER_EPISODE]))
   

In [ ]:
#my_RL_agent_agv.model.save('model_agv.h5')

In [ ]:
import os

base_path = './models/agv_V'
version = 1
while os.path.exists(f'{base_path}{version}.keras'):
    version += 1
my_RL_agent_agv.model.save(f'{base_path}{version}.keras')

In [ ]:
pip install tensorflow